# Influence of homophily and heterophily on oversmoothing of GAT and TransformerConv

## 1 High homophily graphs

### 1.1 pubmed dataset

In [2]:
import torch
import torch_geometric

c:\Users\Mislav.FERIT-PC\Desktop\oversmoothing-GAT-TransformerConv\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] device in use {device}")

[INFO] device in use cuda


In [4]:
path_pubmed = "./data/PubMed"

In [5]:
pubmed_dataset = torch_geometric.datasets.Planetoid(
    root=path_pubmed,
    name="PubMed",
    transform=torch_geometric.transforms.NormalizeFeatures()
)

In [6]:
print("Dataset information")
print("===================")
print(f"Number of graphs in dataset: {len(pubmed_dataset)}")
print(f"Number of features: {pubmed_dataset.num_features}")
print(f"Number of classes: {pubmed_dataset.num_classes}")

Dataset information
Number of graphs in dataset: 1
Number of features: 500
Number of classes: 3


In [7]:
pubmed_data = pubmed_dataset[0]
print(pubmed_data)

Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])


In [8]:
print("Graph information")
print("==================")
print(f"Number of nodes: {pubmed_data.num_nodes}")
print(f"Number of edges: {pubmed_data.num_edges}")
print(f"Has isolated nodes: {pubmed_data.has_isolated_nodes()}")
print(f"Has self loops: {pubmed_data.has_self_loops()}")

Graph information
Number of nodes: 19717
Number of edges: 88648
Has isolated nodes: False
Has self loops: False


## 2 GAT network definition

In [ ]:
class GATNetwork(torch.nn.Module):
    def __init__(
        self,
        number_of_layers: int,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_heads: int,
        dropout_rate: float = 0.5
    ):
        super().__init__()

        self.layers = torch.nn.ModuleList()

        input_channels = input_size
        output_channels = hidden_size
        for i in range(number_of_layers - 1):
            self.layers.append(
                torch_geometric.nn.LayerNorm(in_channels=input_channels)
            )
            self.layers.append(
                torch_geometric.nn.GATv2Conv(in_channels=input_channels, out_channels=output_channels, heads=num_heads)
            )
            self.layers.append(
                torch.nn.Dropout(p=dropout_rate)
            )
            self.layers.append(
                torch.nn.GELU()
            )
            input_channels = hidden_size * num_heads

        output_channels = output_size
        self.layers.append(
            torch_geometric.nn.LayerNorm(in_channels=input_channels)
        )
        self.layers.append(torch_geometric.nn.GATv2Conv(in_channels=input_channels, out_channels=output_channels, heads=num_heads, concat=False))
        
        self.layers = torch.nn.ModuleList(self.layers)
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        for layer in self.layers:
            if isinstance(layer, torch_geometric.nn.GATv2Conv):
                x = layer(x, edge_index)
            else:
                x = layer(x)
        return x

## 3 TransformerConv network definition

In [ ]:
class TransformerConvNetwork(torch.nn.Module):
    def __init__(
        self,
        number_of_layers: int,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_heads: int,
        dropout_rate: float = 0.5
    ):
        super().__init__()

        self.layers = torch.nn.ModuleList()

        input_channels = input_size
        output_channels = hidden_size
        for i in range(number_of_layers - 1):
            self.layers.append(
                torch_geometric.nn.LayerNorm(in_channels=input_channels)
            )
            self.layers.append(
                torch_geometric.nn.TransformerConv(in_channels=input_channels, out_channels=output_channels, heads=num_heads, beta=True)
            )
            self.layers.append(
                torch.nn.Dropout(p=dropout_rate),
            )
            self.layers.append(
                torch.nn.GELU()
            )
            input_channels = hidden_size * num_heads
            
        output_channels = output_size
        self.layers.append(
            torch_geometric.nn.LayerNorm(in_channels=input_channels)
        )
        self.layers.append(torch_geometric.nn.TransformerConv(
            in_channels = input_channels,
            out_channels = output_channels,
            heads = num_heads,
            concat = False,
            beta = True
        ))
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        for layer in self.layers:
            if isinstance(layer, torch_geometric.nn.TransformerConv):
                x = layer(x, edge_index)
            else:
                x = layer(x)
        return x


# 4 Transductive learning setting

In [11]:
class TransductiveTrainer:
    def __init__(
        self,
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        loss_function: torch.nn.Module,
        data: torch_geometric.data.Data,
        epochs: int
    ):
        self.model = model
        self.optimizer = optimizer
        self.loss_function = loss_function
        self.data = data
        self.epochs = epochs
        self.logger = {
            "train_loss": [],
            "train_accuracy": [],
            "val_loss": [],
            "val_accuracy": []
        }

    def fit(self):
        for epoch in range(self.epochs):
            # training step
            self.model.train()
            logits = self.model(self.data.x, self.data.edge_index)
            train_loss_value = self.loss_function(logits[self.data.train_mask], self.data.y[self.data.train_mask])
            self.optimizer.zero_grad()
            train_loss_value.backward()
            self.optimizer.step()
            train_accuracy = self.accuracy(logits[self.data.train_mask].argmax(dim=1), self.data.y[self.data.train_mask])

            train_loss_value = train_loss_value.detach().cpu().numpy()
            train_accuracy = train_accuracy.detach().cpu().numpy()
            print(f"[INFO] epoch {epoch + 1}:\n\ttraining loss: {train_loss_value}")
            print(f"\n\ttraining accuracy: {train_accuracy}")
            self.logger["train_loss"].append(train_loss_value)
            self.logger["train_accuracy"].append(train_accuracy)

            # validation step
            self.model.eval()
            with torch.no_grad():
                logits = self.model(self.data.x, self.data.edge_index)
                val_loss_value = self.loss_function(logits[self.data.val_mask], self.data.y[self.data.val_mask])
                val_accuracy = self.accuracy(logits[self.data.val_mask].argmax(dim=1), self.data.y[self.data.val_mask])

                val_loss_value = val_loss_value.cpu().numpy()
                val_accuracy = val_accuracy.cpu().numpy()
                print(f"\n\tval loss: {val_loss_value}")
                print(f"\n\tval accuracy: {val_accuracy}")
                print(f"\n")
                self.logger["val_loss"].append(val_loss_value)
                self.logger["val_accuracy"].append(val_accuracy)
    
    def accuracy(self, y_predict: torch.Tensor, y_truth: torch.Tensor):
        return torch.sum(y_predict == y_truth) / len(y_truth)
    
    @torch.no_grad()
    def test(self, y_predict: torch.Tensor | None = None, y_truth: torch.Tensor | None = None):
        self.model.eval()
        if y_predict is None and y_truth is None:
            logits = self.model(self.data.x, self.data.edge_index)
            y_predict = logits[self.data.test_mask].argmax(dim=1)
            y_truth = self.data.y[self.data.test_mask]
        test_accuracy = self.accuracy(y_predict, y_truth)

        return test_accuracy.cpu().numpy()